In [3]:
!pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   - -------------------------------------- 1.6/48.9 MB 12.0 MB/s eta 0:00:04
   -- ------------------------------------- 2.9/48.9 MB 6.7 MB/s eta 0:00:07
   ---- ----------------------------------- 5.8/48.9 MB 9.3 MB/s eta 0:00:05
   ------- -------------------------------- 8.9/48.9 MB 10.1 MB/s eta 0:00:04
   -------- ------------------------------- 10.2/48.9 MB 9.2 MB/s eta 0:00:05
   ---------- ----------------------------- 13.1/48.9 MB 9.7 MB/s eta 0:00:04
   ------------- -------------------------- 16.3/48.9 MB 10.3 MB/s eta 0:00:04
   ---------------- ----------------------- 19.9/48.9 MB 10.9 MB/s eta 0:00:03
   ------------------ --------------------- 22.8/48.9 MB 11.0 MB/s eta 0:00:03
   --------------------- ------------------ 26.0/48.9 MB 11.3 MB/s eta 0:00:03
   ----------------------- ---------------- 29.1/48.9 MB 11.6 MB/s eta 0:00:02
   -------------------------- ------------- 33.0/48.9 MB 11.9 MB/s et

In [5]:
import xgboost as xgb
print(xgb.__version__)

3.4.1


In [7]:
import os
print(os.getcwd())
print([f for f in os.listdir('.') if 'xgb' in f or 'scaled' in f])

C:\Users\User
['test_scaled.csv', 'test_scaled_v2.csv', 'test_xgb_features.csv', 'test_xgb_features_v2.csv', 'train_scaled.csv', 'train_scaled_v2.csv', 'train_xgb_features.csv', 'train_xgb_features_v2.csv', 'val_scaled.csv', 'val_scaled_v2.csv', 'val_xgb_features.csv', 'val_xgb_features_v2.csv']


In [9]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load the XGBoost-ready features
train = pd.read_csv('train_xgb_features_v2.csv')
val = pd.read_csv('val_xgb_features_v2.csv')
test = pd.read_csv('test_xgb_features_v2.csv')

# Define the feature set — everything except identifiers, datetime, and target
exclude_cols = ['datetime', 'place', 'city', 'state', 'solar_radiation']
feature_cols = [c for c in train.columns if c not in exclude_cols]
print(f"Feature count: {len(feature_cols)}")
print(feature_cols)

X_train, y_train = train[feature_cols], train['solar_radiation']
X_val, y_val = val[feature_cols], val['solar_radiation']
X_test, y_test = test[feature_cols], test['solar_radiation']

# Use DMatrix — more memory-efficient than raw DataFrames for training
dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=False)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

# Baseline params — a reasonable starting point before hyperparameter search (Section 3.5.4)
params = {
    'objective': 'reg:squarederror',
    'max_depth': 6,
    'eta': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42,
}

evals = [(dtrain, 'train'), (dval, 'val')]
model = xgb.train(
    params, dtrain,
    num_boost_round=500,
    evals=evals,
    early_stopping_rounds=20,
    verbose_eval=50
)

# Evaluate on test set
y_pred = model.predict(dtest)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nTest RMSE: {rmse:.4f}")
print(f"Test MAE:  {mae:.4f}")
print(f"Test R²:   {r2:.4f}")

model.save_model('xgb_baseline_v1.json')
print("\nModel saved to xgb_baseline_v1.json")

Feature count: 31
['temperature', 'pressure', 'dew_point', 'humidity', 'wind_speed', 'wind_chill', 'uv_index', 'solar_radiation_lag_1', 'temperature_lag_1', 'uv_index_lag_1', 'solar_radiation_lag_2', 'temperature_lag_2', 'uv_index_lag_2', 'solar_radiation_lag_3', 'temperature_lag_3', 'uv_index_lag_3', 'solar_radiation_lag_24', 'temperature_lag_24', 'uv_index_lag_24', 'solar_radiation_roll_mean_3h', 'solar_radiation_roll_std_3h', 'solar_radiation_roll_mean_24h', 'solar_radiation_roll_std_24h', 'temperature_roll_mean_3h', 'temperature_roll_std_3h', 'temperature_roll_mean_24h', 'temperature_roll_std_24h', 'hour_of_day', 'month_of_year', 'is_northeast_monsoon', 'is_southwest_monsoon']
[0]	train-rmse:0.90237	val-rmse:0.91820
[50]	train-rmse:0.10407	val-rmse:0.12322
[100]	train-rmse:0.09716	val-rmse:0.12052
[150]	train-rmse:0.09353	val-rmse:0.11929
[200]	train-rmse:0.09134	val-rmse:0.11876
[250]	train-rmse:0.08939	val-rmse:0.11807
[300]	train-rmse:0.08813	val-rmse:0.11767
[350]	train-rmse:0.

In [11]:
importance = model.get_score(importance_type='gain')
importance_df = pd.DataFrame(list(importance.items()), columns=['feature', 'gain'])
importance_df = importance_df.sort_values('gain', ascending=False)
print(importance_df.head(15).to_string(index=False))

                      feature        gain
       solar_radiation_lag_24 4174.607910
        solar_radiation_lag_1  947.059204
                  hour_of_day  188.094559
  solar_radiation_roll_std_3h  136.645309
                     uv_index  126.851852
 solar_radiation_roll_mean_3h  121.348274
        solar_radiation_lag_3   58.501812
solar_radiation_roll_mean_24h   27.540714
               uv_index_lag_3   27.103210
               uv_index_lag_2   20.217381
              uv_index_lag_24   16.672310
 solar_radiation_roll_std_24h   13.713920
            temperature_lag_3    7.150165
               uv_index_lag_1    5.901230
           temperature_lag_24    4.877075


In [13]:
# Persistence baseline: predict lag_1 directly as the forecast
persistence_pred = test['solar_radiation_lag_1'].values
persistence_rmse = np.sqrt(mean_squared_error(y_test, persistence_pred))
persistence_mae = mean_absolute_error(y_test, persistence_pred)
persistence_r2 = r2_score(y_test, persistence_pred)

print(f"Persistence baseline — RMSE: {persistence_rmse:.4f}, MAE: {persistence_mae:.4f}, R²: {persistence_r2:.4f}")
print(f"XGBoost              — RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")
print(f"\nRMSE improvement over persistence: {(1 - rmse/persistence_rmse)*100:.1f}%")

Persistence baseline — RMSE: 0.3506, MAE: 0.2228, R²: 0.8766
XGBoost              — RMSE: 0.1184, MAE: 0.0573, R²: 0.9859

RMSE improvement over persistence: 66.2%


In [1]:
import pickle

with open('station_scalers_v2.pkl', 'rb') as f:
    scalers = pickle.load(f)

# Each entry has scaler.mean_ / scaler.scale_ for the 7 "complete" columns,
# plus a separate uv_mean/uv_std for uv_index.
# solar_radiation's position within complete_cols tells us which mean/std to use.
complete_cols = ['temperature', 'pressure', 'dew_point', 'humidity',
                  'wind_speed', 'wind_chill', 'solar_radiation']
solar_idx = complete_cols.index('solar_radiation')

example_place = list(scalers.keys())[0]
solar_mean = scalers[example_place]['scaler'].mean_[solar_idx]
solar_std = scalers[example_place]['scaler'].scale_[solar_idx]
print(f"Example station '{example_place}': solar_radiation mean={solar_mean:.2f}, std={solar_std:.2f}")

Example station 'Air Itam': solar_radiation mean=210.33, std=286.02


In [3]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

with open('station_scalers_v2.pkl', 'rb') as f:
    scalers = pickle.load(f)

complete_cols = ['temperature', 'pressure', 'dew_point', 'humidity',
                  'wind_speed', 'wind_chill', 'solar_radiation']
solar_idx = complete_cols.index('solar_radiation')

test = pd.read_csv('test_xgb_features_v2.csv')
test['y_true_scaled'] = test['solar_radiation']
test['y_pred_scaled'] = y_pred  # from your earlier XGBoost prediction

# Inverse-transform per station
def inverse_solar(row, col):
    place = row['place']
    mean = scalers[place]['scaler'].mean_[solar_idx]
    std = scalers[place]['scaler'].scale_[solar_idx]
    return row[col] * std + mean

test['y_true_real'] = test.apply(lambda r: inverse_solar(r, 'y_true_scaled'), axis=1)
test['y_pred_real'] = test.apply(lambda r: inverse_solar(r, 'y_pred_scaled'), axis=1)

# Recompute metrics in real W/m² units
rmse_real = np.sqrt(mean_squared_error(test['y_true_real'], test['y_pred_real']))
mae_real = mean_absolute_error(test['y_true_real'], test['y_pred_real'])
r2_real = r2_score(test['y_true_real'], test['y_pred_real'])

print(f"RMSE (real units, W/m²): {rmse_real:.2f}")
print(f"MAE  (real units, W/m²): {mae_real:.2f}")
print(f"R²   (real units):       {r2_real:.4f}")

# nRMSE — RMSE normalized by the mean of the true values (per 3.6.1)
nrmse = rmse_real / test['y_true_real'].mean()
print(f"nRMSE: {nrmse:.4f} ({nrmse*100:.2f}%)")

# MAPE — careful here: solar_radiation is 0 at night, so a naive MAPE would
# divide by zero / blow up. Standard practice: exclude near-zero true values.
daytime_mask = test['y_true_real'] > 10  # exclude near-zero (nighttime) readings
mape = (np.abs((test.loc[daytime_mask, 'y_true_real'] - test.loc[daytime_mask, 'y_pred_real'])
               / test.loc[daytime_mask, 'y_true_real'])).mean() * 100
print(f"MAPE (daytime only, true>10 W/m²): {mape:.2f}%")

test.to_csv('test_predictions_real_units.csv', index=False)

NameError: name 'y_pred' is not defined

In [5]:
import pandas as pd
import numpy as np
import xgboost as xgb
import pickle
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Reload the saved model
model = xgb.Booster()
model.load_model('xgb_baseline_v1.json')

# Reload test data and rebuild the feature matrix exactly as before
test = pd.read_csv('test_xgb_features_v2.csv')
exclude_cols = ['datetime', 'place', 'city', 'state', 'solar_radiation']
feature_cols = [c for c in test.columns if c not in exclude_cols]

X_test = test[feature_cols]
y_test = test['solar_radiation']
dtest = xgb.DMatrix(X_test, label=y_test)

y_pred = model.predict(dtest)
print(f"Predictions generated: {len(y_pred):,}")

# Confirm scaled-unit metrics still match what we saw before, as a sanity check
rmse_scaled = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Scaled RMSE (should be ~0.1184, matching earlier run): {rmse_scaled:.4f}")

Predictions generated: 749,012
Scaled RMSE (should be ~0.1184, matching earlier run): 0.1184


In [7]:
with open('station_scalers_v2.pkl', 'rb') as f:
    scalers = pickle.load(f)

complete_cols = ['temperature', 'pressure', 'dew_point', 'humidity',
                  'wind_speed', 'wind_chill', 'solar_radiation']
solar_idx = complete_cols.index('solar_radiation')

test['y_true_scaled'] = test['solar_radiation']
test['y_pred_scaled'] = y_pred

def inverse_solar(row, col):
    place = row['place']
    mean = scalers[place]['scaler'].mean_[solar_idx]
    std = scalers[place]['scaler'].scale_[solar_idx]
    return row[col] * std + mean

test['y_true_real'] = test.apply(lambda r: inverse_solar(r, 'y_true_scaled'), axis=1)
test['y_pred_real'] = test.apply(lambda r: inverse_solar(r, 'y_pred_scaled'), axis=1)

rmse_real = np.sqrt(mean_squared_error(test['y_true_real'], test['y_pred_real']))
mae_real = mean_absolute_error(test['y_true_real'], test['y_pred_real'])
r2_real = r2_score(test['y_true_real'], test['y_pred_real'])

print(f"RMSE (real units, W/m²): {rmse_real:.2f}")
print(f"MAE  (real units, W/m²): {mae_real:.2f}")
print(f"R²   (real units):       {r2_real:.4f}")

nrmse = rmse_real / test['y_true_real'].mean()
print(f"nRMSE: {nrmse:.4f} ({nrmse*100:.2f}%)")

daytime_mask = test['y_true_real'] > 10
mape = (np.abs((test.loc[daytime_mask, 'y_true_real'] - test.loc[daytime_mask, 'y_pred_real'])
               / test.loc[daytime_mask, 'y_true_real'])).mean() * 100
print(f"MAPE (daytime only, true>10 W/m²): {mape:.2f}%")

test.to_csv('test_predictions_real_units.csv', index=False)
print("\nSaved test_predictions_real_units.csv")

RMSE (real units, W/m²): 32.91
MAE  (real units, W/m²): 15.94
R²   (real units):       0.9860
nRMSE: 0.1590 (15.90%)
MAPE (daytime only, true>10 W/m²): 13.00%

Saved test_predictions_real_units.csv


In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
import pickle
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Reload model
model = xgb.Booster()
model.load_model('xgb_baseline_v1.json')

# Reload test data + rebuild features
test = pd.read_csv('test_xgb_features_v2.csv')
exclude_cols = ['datetime', 'place', 'city', 'state', 'solar_radiation']
feature_cols = [c for c in test.columns if c not in exclude_cols]
X_test = test[feature_cols]
y_test = test['solar_radiation']
dtest = xgb.DMatrix(X_test, label=y_test)
y_pred = model.predict(dtest)

# Reload scalers and inverse-transform
with open('station_scalers_v2.pkl', 'rb') as f:
    scalers = pickle.load(f)

complete_cols = ['temperature', 'pressure', 'dew_point', 'humidity',
                  'wind_speed', 'wind_chill', 'solar_radiation']
solar_idx = complete_cols.index('solar_radiation')

test['y_true_scaled'] = test['solar_radiation']
test['y_pred_scaled'] = y_pred

def inverse_solar(row, col):
    place = row['place']
    mean = scalers[place]['scaler'].mean_[solar_idx]
    std = scalers[place]['scaler'].scale_[solar_idx]
    return row[col] * std + mean

test['y_true_real'] = test.apply(lambda r: inverse_solar(r, 'y_true_scaled'), axis=1)
test['y_pred_real'] = test.apply(lambda r: inverse_solar(r, 'y_pred_scaled'), axis=1)

print(f"Reloaded and rebuilt. Rows: {len(test):,}")
print(f"RMSE check: {np.sqrt(mean_squared_error(test['y_true_real'], test['y_pred_real'])):.2f} "
      f"(should be ~32.91)")

Reloaded and rebuilt. Rows: 749,012
RMSE check: 32.91 (should be ~32.91)


In [5]:
station_metrics = test.groupby('place').apply(
    lambda g: pd.Series({
        'rmse': np.sqrt(mean_squared_error(g['y_true_real'], g['y_pred_real'])),
        'mae': mean_absolute_error(g['y_true_real'], g['y_pred_real']),
        'r2': r2_score(g['y_true_real'], g['y_pred_real']),
        'n_rows': len(g)
    }), include_groups=False
).sort_values('rmse', ascending=False)

print("Worst 10 stations by RMSE:")
print(station_metrics.head(10).to_string())
print("\nBest 10 stations by RMSE:")
print(station_metrics.tail(10).to_string())
print(f"\nRMSE range across stations: {station_metrics['rmse'].min():.2f} to {station_metrics['rmse'].max():.2f}")
print(f"R² range across stations: {station_metrics['r2'].min():.4f} to {station_metrics['r2'].max():.4f}")

station_metrics.to_csv('station_level_metrics.csv')
print("\nSaved station_level_metrics.csv")

Worst 10 stations by RMSE:
                         rmse        mae        r2   n_rows
place                                                      
Sandakan            38.014209  17.710554  0.982159  26343.0
Bandar Putra Kulai  37.729707  17.631447  0.979108  27334.0
Bangsar             37.116822  17.927010  0.980254    519.0
Kudat               36.551985  17.394667  0.983883  26167.0
Sibu Jaya           36.547317  17.884398  0.982487  26528.0
Tawau               36.209979  17.525732  0.981634  26854.0
Kota Sentosa        35.279045  17.451635  0.983860  28861.0
Pekan Sibu          35.215304  17.771944  0.983129   1989.0
Sabu                35.198470  17.159558  0.982613  25152.0
Taman Kulai Besar   35.155113  17.705047  0.981200   3905.0

Best 10 stations by RMSE:
                                           rmse        mae        r2  n_rows
place                                                                       
Lojing                                20.480746  10.233520  0.994004  11

In [7]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Use the RAW (unscaled) splits — solar_radiation here is already in real W/m² units
test_raw = pd.read_csv('test_raw_v2.csv')
test_raw['datetime'] = pd.to_datetime(test_raw['datetime'])
test_raw = test_raw.sort_values(['place', 'datetime']).reset_index(drop=True)

# Persistence: predict next hour = same as previous hour, per station
test_raw['solar_radiation_lag_1'] = test_raw.groupby('place')['solar_radiation'].shift(1)
valid = test_raw.dropna(subset=['solar_radiation_lag_1'])

rmse_p = np.sqrt(mean_squared_error(valid['solar_radiation'], valid['solar_radiation_lag_1']))
mae_p = mean_absolute_error(valid['solar_radiation'], valid['solar_radiation_lag_1'])
r2_p = r2_score(valid['solar_radiation'], valid['solar_radiation_lag_1'])

print(f"Persistence baseline (real units, W/m²):")
print(f"  RMSE: {rmse_p:.2f}")
print(f"  MAE:  {mae_p:.2f}")
print(f"  R²:   {r2_p:.4f}")

Persistence baseline (real units, W/m²):
  RMSE: 98.79
  MAE:  62.35
  R²:   0.8738


In [9]:
!pip install pmdarima prophet

   ---------------------------------------- 0.0/715.6 kB ? eta -:--:--
   -------------- ------------------------- 262.1/715.6 kB ? eta -:--:--
   ---------------------------------------- 715.6/715.6 kB 2.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.1 MB 3.7 MB/s eta 0:00:04
   ----- ---------------------------------- 1.6/12.1 MB 4.4 MB/s eta 0:00:03
   ------- -------------------------------- 2.4/12.1 MB 3.5 MB/s eta 0:00:03
   -------- ------------------------------- 2.6/12.1 MB 3.2 MB/s eta 0:00:03
   ---------- ----------------------------- 3.1/12.1 MB 2.8 MB/s eta 0:00:04
   ----------- ---------------------------- 3.4/12.1 MB 2.7 MB/s eta 0:00:04
   ------------ --------------------------- 3.7/12.1 MB 2.4 MB/s eta 0:00:04
   ------------ --------------------------- 3.9/12.1 MB 2.2 MB/s eta 0:00:04
   -------------- ------------------------- 4.5/12.1 MB 2.3 MB/s eta 0:00:04
   ----------

In [13]:
import pandas as pd
import numpy as np
import time
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings('ignore')  # SARIMA throws a lot of convergence warnings; suppress for readability

train_raw = pd.read_csv('train_raw_v2.csv')
test_raw = pd.read_csv('test_raw_v2.csv')
for d in [train_raw, test_raw]:
    d['datetime'] = pd.to_datetime(d['datetime'])

sample_stations = train_raw['place'].unique()[:3]
print(f"Testing on: {list(sample_stations)}")

sarima_results = {}
for place in sample_stations:
    tr = train_raw[train_raw['place'] == place].sort_values('datetime').set_index('datetime')
    te = test_raw[test_raw['place'] == place].sort_values('datetime').set_index('datetime')

    # Give the series an explicit hourly frequency so statsmodels can forecast forward correctly
    tr_series = tr['solar_radiation'].asfreq('h')

    start = time.time()
    model = SARIMAX(tr_series, order=(1,0,1), seasonal_order=(1,0,1,24),
                     enforce_stationarity=False, enforce_invertibility=False)
    fit = model.fit(disp=False)
    elapsed = time.time() - start

    forecast = fit.forecast(steps=len(te))
    rmse = np.sqrt(mean_squared_error(te['solar_radiation'].values, forecast.values))

    sarima_results[place] = {'time_sec': elapsed, 'rmse': rmse}
    print(f"{place}: fit took {elapsed:.1f}s, RMSE={rmse:.2f}")

avg_time = np.mean([v['time_sec'] for v in sarima_results.values()])
print(f"\nAverage fit time per station: {avg_time:.1f}s")
print(f"Estimated time for all 71 stations: {avg_time * 71 / 60:.1f} minutes")

Testing on: ['Air Itam', 'Ampang Jaya', 'Ayer Keroh']
Air Itam: fit took 321.9s, RMSE=210.66
Ampang Jaya: fit took 235.0s, RMSE=398.66
Ayer Keroh: fit took 376.2s, RMSE=342.16

Average fit time per station: 311.1s
Estimated time for all 71 stations: 368.1 minutes


In [3]:
import pandas as pd
import numpy as np
import time
import warnings
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')

place = 'Air Itam'

train_raw = pd.read_csv('train_raw_v2.csv')
test_raw = pd.read_csv('test_raw_v2.csv')
for d in [train_raw, test_raw]:
    d['datetime'] = pd.to_datetime(d['datetime'])

tr = train_raw[train_raw['place'] == place].sort_values('datetime').set_index('datetime')
te = test_raw[test_raw['place'] == place].sort_values('datetime').set_index('datetime')

# Hourly grid; gaps become NaN (SARIMAX tolerates NaN in the history)
tr_series = tr['solar_radiation'].asfreq('h')
te_series = te['solar_radiation'].asfreq('h')

# Speed-up: fit on only the most recent 2 years of training data
tr_series = tr_series.iloc[-24 * 365 * 2:]

start = time.time()
model = SARIMAX(tr_series, order=(1, 0, 1), seasonal_order=(1, 0, 1, 24),
                enforce_stationarity=False, enforce_invertibility=False)
fit = model.fit(disp=False)
print(f"Fit took {time.time() - start:.1f}s")

# Forecast 24h at a time, then reveal the true values before the next day
horizon = 24
preds = []
for i in range(0, len(te_series), horizon):
    steps = min(horizon, len(te_series) - i)
    preds.extend(fit.forecast(steps=steps).values)
    fit = fit.append(te_series.iloc[i:i + steps], refit=False)

preds = np.array(preds[:len(te_series)])
actual = te_series.values

# Score only hours where a real reading exists (skip the gap hours)
mask = ~np.isnan(actual)
rmse = np.sqrt(mean_squared_error(actual[mask], preds[mask]))
mae = mean_absolute_error(actual[mask], preds[mask])
print(f"{place} — rolling 24h SARIMA: RMSE={rmse:.2f}, MAE={mae:.2f}  (total time {time.time() - start:.1f}s)")

Fit took 326.3s


ValueError: Given `endog` does not have an index that extends the index of the model.

In [5]:
print("train ends:  ", tr_series.index[-1])
print("test starts: ", te_series.index[0])
print("expected:    ", tr_series.index[-1] + pd.Timedelta(hours=1))

train ends:   2023-04-12 15:00:00
test starts:  2023-12-21 17:00:00
expected:     2023-04-12 16:00:00


In [1]:
import pandas as pd
import numpy as np
import time
import warnings
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')
place = 'Air Itam'

def load(name):
    d = pd.read_csv(name)
    d['datetime'] = pd.to_datetime(d['datetime'])
    return d[d['place'] == place].sort_values('datetime').set_index('datetime')['solar_radiation']

tr, va, te = load('train_raw_v2.csv'), load('val_raw_v2.csv'), load('test_raw_v2.csv')

# Fit on the last 2 years of training data (for speed)
tr_series = tr.asfreq('h').iloc[-24 * 365 * 2:]

# Hourly grid covering validation + test, starting right after training ends
future_index = pd.date_range(tr_series.index[-1] + pd.Timedelta(hours=1), te.index.max(), freq='h')
future = pd.concat([va, te]).reindex(future_index)

start = time.time()
fit = SARIMAX(tr_series, order=(1, 0, 1), seasonal_order=(1, 0, 1, 24),
              enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
print(f"Fit took {time.time() - start:.1f}s")

# Forecast one day, reveal the truth, repeat (through validation, then test)
horizon = 24
preds = []
for i in range(0, len(future), horizon):
    chunk = future.iloc[i:i + horizon]
    preds.extend(fit.forecast(steps=len(chunk)).values)
    fit = fit.append(chunk, refit=False)
preds = pd.Series(preds, index=future.index)

# Score on the TEST period only, real readings only
sarima_pred = preds.reindex(te.index).values
rmse = np.sqrt(mean_squared_error(te.values, sarima_pred))
mae = mean_absolute_error(te.values, sarima_pred)

# Persistence on the same station and rows, for a fair comparison
te_h = te.asfreq('h')
pers = te_h.shift(1)
m = te_h.notna() & pers.notna()
pers_rmse = np.sqrt(mean_squared_error(te_h[m], pers[m]))

print(f"\n{place} (test period)")
print(f"  SARIMA (rolling 24h): RMSE={rmse:.2f}, MAE={mae:.2f}")
print(f"  Persistence:          RMSE={pers_rmse:.2f}")
print(f"Total time: {time.time() - start:.1f}s")

Fit took 166.8s

Air Itam (test period)
  SARIMA (rolling 24h): RMSE=66.24, MAE=41.29
  Persistence:          RMSE=102.49
Total time: 1522.9s


In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')

STATIONS = ['Air Itam', 'Sandakan', 'Kudat', 'Johor Bahru', 'Saujana Villa']
OUT = 'sarima_results.csv'

def load(name, place):
    d = pd.read_csv(name, usecols=['place', 'datetime', 'solar_radiation'])
    d['datetime'] = pd.to_datetime(d['datetime'])
    d = d[d['place'] == place].sort_values('datetime')
    return d.set_index('datetime')['solar_radiation']

done = set(pd.read_csv(OUT)['place']) if os.path.exists(OUT) else set()

for place in STATIONS:
    if place in done:
        print(f"Skipping {place} (already done)")
        continue
    t0 = time.time()
    tr, va, te = (load(f, place) for f in
                  ['train_raw_v2.csv', 'val_raw_v2.csv', 'test_raw_v2.csv'])

    tr_series = tr.asfreq('h').iloc[-24 * 365 * 2:]
    future_index = pd.date_range(tr_series.index[-1] + pd.Timedelta(hours=1),
                                 te.index.max(), freq='h')
    future = pd.concat([va, te]).reindex(future_index)

    fit = SARIMAX(tr_series, order=(1, 0, 1), seasonal_order=(1, 0, 1, 24),
                  enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

    preds = []
    for i in range(0, len(future), 24):
        chunk = future.iloc[i:i + 24]
        preds.extend(fit.forecast(steps=len(chunk)).values)
        fit = fit.append(chunk, refit=False)
    preds = pd.Series(preds, index=future.index)

    sarima_pred = preds.reindex(te.index).values
    te_h = te.asfreq('h')
    pers = te_h.shift(1)
    m = te_h.notna() & pers.notna()

    row = pd.DataFrame([{
        'place': place,
        'sarima_rmse': np.sqrt(mean_squared_error(te.values, sarima_pred)),
        'sarima_mae': mean_absolute_error(te.values, sarima_pred),
        'persistence_rmse': np.sqrt(mean_squared_error(te_h[m], pers[m])),
        'minutes': (time.time() - t0) / 60,
    }])
    row.to_csv(OUT, mode='a', header=not os.path.exists(OUT), index=False)
    print(f"{place} done in {(time.time() - t0) / 60:.1f} min")

Air Itam done in 25.6 min


In [1]:
import pandas as pd
import numpy as np
import pickle

W = 24  # window length

for split in ['val', 'test']:
    df = pd.read_csv(f'{split}_scaled_v2.csv', usecols=['place', 'datetime'])
    df['datetime'] = pd.to_datetime(df['datetime'])
    rows = []
    for place, g in df.groupby('place'):          # same order as build_sequences
        times = g.sort_values('datetime')['datetime'].values
        n = len(times)
        if n <= W:
            continue
        hours = np.diff(times).astype('timedelta64[h]').astype(int)
        bad = np.concatenate([[0], np.cumsum(hours != 1)])
        i = np.arange(n - W)
        valid = (bad[i + W] - bad[i]) == 0        # no gap inside the window
        for t in times[i[valid] + W]:
            rows.append((place, t))
    meta = pd.DataFrame(rows, columns=['place', 'datetime'])
    y_len = np.load(f'{split}_y_{W}h_v2.npy', mmap_mode='r').shape[0]
    print(f"{split}: meta={len(meta):,}, y={y_len:,}  -> {'MATCH' if len(meta) == y_len else 'MISMATCH'}")
    meta.to_csv(f'{split}_meta_{W}h.csv', index=False)

# Per-station solar scaling stats, so Kaggle doesn't need the pickle
with open('station_scalers_v2.pkl', 'rb') as f:
    scalers = pickle.load(f)
idx = ['temperature', 'pressure', 'dew_point', 'humidity',
       'wind_speed', 'wind_chill', 'solar_radiation'].index('solar_radiation')
pd.DataFrame([{'place': p, 'solar_mean': s['scaler'].mean_[idx], 'solar_std': s['scaler'].scale_[idx]}
              for p, s in scalers.items()]).to_csv('solar_scaler_stats.csv', index=False)
print("Saved solar_scaler_stats.csv")

val: meta=741,034, y=741,034  -> MATCH
test: meta=735,451, y=735,451  -> MATCH
Saved solar_scaler_stats.csv


In [1]:
import pandas as pd, numpy as np, pickle

W = 24
df = pd.read_csv('test_scaled_v2.csv', usecols=['place', 'datetime', 'solar_radiation'])
df['datetime'] = pd.to_datetime(df['datetime'])
with open('station_scalers_v2.pkl', 'rb') as f:
    scalers = pickle.load(f)
solar_idx = 6  # position of solar_radiation among the 7 scaled columns

rows, y_check = [], []
for place, g in df.groupby('place'):
    g = g.sort_values('datetime').reset_index(drop=True)
    t = g['datetime'].values
    ok = (np.diff(t).astype('timedelta64[h]').astype(int) == 1).astype(int)
    csum = np.concatenate([[0], np.cumsum(ok)])
    i = np.arange(0, len(g) - W)
    tgt = i[(csum[i + W] - csum[i]) == W] + W
    sc = scalers[place]['scaler']
    rows.append(pd.DataFrame({'place': place, 'datetime': t[tgt],
                              'sol_mean': sc.mean_[solar_idx],
                              'sol_std': sc.scale_[solar_idx]}))
    y_check.append(g['solar_radiation'].values[tgt])

meta = pd.concat(rows, ignore_index=True)
y_saved = np.load('test_y_24h_v2.npy')
print(len(meta), len(y_saved))
assert len(meta) == len(y_saved)
assert np.allclose(np.concatenate(y_check), y_saved, atol=1e-5)
meta.to_csv('test_meta_24h_v2.csv', index=False)
print("Aligned and saved.")

735451 735451
Aligned and saved.


In [1]:
import pandas as pd

audit = pd.read_csv('station_audit.csv')
prev = set(audit[audit['temperature_missing_pct'] < 20]['place'])
print(f"Quality filter: {len(prev)} stations")

stages = [
    ('weather_hourly_final.csv',    'after hourly rounding'),
    ('weather_no_gaps.csv',         'after removing gaps >3h'),
    ('weather_final_clean_v2.csv',  'final clean file'),
    ('train_raw_v2.csv',            'train split'),
    ('val_raw_v2.csv',              'val split'),
    ('test_raw_v2.csv',             'test split'),
    ('test_meta_24h.csv',           'test windows (24h)'),
]
for path, label in stages:
    s = set(pd.read_csv(path, usecols=['place'])['place'].unique())
    lost = sorted(prev - s)
    print(f"{label:26s} {len(s)} stations" + (f"  | lost: {lost}" if lost else ""))
    prev = s

Quality filter: 71 stations
after hourly rounding      71 stations
after removing gaps >3h    67 stations  | lost: ['Bandar Sunway', 'Sungai Buloh', 'Taman Eng Ann', 'Taman Politeknik']
final clean file           67 stations
train split                67 stations
val split                  67 stations
test split                 67 stations
test windows (24h)         67 stations


In [1]:
import pandas as pd

lost = ['Bandar Sunway', 'Sungai Buloh', 'Taman Eng Ann', 'Taman Politeknik']
cols = ['temperature', 'pressure', 'dew_point', 'humidity',
        'wind_speed', 'wind_chill', 'uv_index', 'solar_radiation']

df = pd.read_csv('weather_interpolated.csv', usecols=['place'] + cols,
                 dtype={'place': 'category'})
df = df[df['place'].isin(lost)]

print(df.groupby('place', observed=True).size().rename('rows').to_string())
print("\n% missing per column:")
print((df.groupby('place', observed=True)[cols]
         .apply(lambda g: g.isnull().mean() * 100).round(1)).to_string())

place
Bandar Sunway       31602
Sungai Buloh        70428
Taman Eng Ann       58385
Taman Politeknik    12472

% missing per column:
                  temperature  pressure  dew_point  humidity  wind_speed  wind_chill  uv_index  solar_radiation
place                                                                                                          
Bandar Sunway             0.0       0.0        0.0       0.0       100.0         0.0       0.0              0.0
Sungai Buloh              1.8      35.2        0.8       0.2       100.0        48.7       0.0              0.0
Taman Eng Ann            12.3       0.0       12.4      11.7       100.0        39.2       0.0              0.0
Taman Politeknik          0.2       0.0       56.1     100.0         0.2         0.2       0.0              0.0
